In [1]:
# 1: Install and import dependencies
%pip -q install datasets requests tqdm mwparserfromhell nltk lxml

import os
import re
import json
import html
import bz2
import shutil
import sqlite3
import subprocess
from pathlib import Path
from typing import Dict, Any, Iterator, List, Tuple, Set, Optional
from collections import Counter

import requests
from tqdm.auto import tqdm
from datasets import load_dataset

import mwparserfromhell
import nltk
from lxml import etree

nltk.download("punkt", quiet=True)


True

In [2]:
# 2: Download the 2WikiMultiHopQA DEV split (DEV == 'validation' on HF)

PREFERRED_DATASET = "framolfese/2WikiMultihopQA"  # HotpotQA-like schema
FALLBACK_DATASET  = "xanhho/2WikiMultihopQA"      # Official mirror on HF

def load_2wikimultihopqa_dev() -> Tuple[List[Dict[str, Any]], str]:
    """
    Load DEV (validation) split from Hugging Face.
    Returns:
        records: list of dict examples
        source:  dataset id used
    """
    for ds_name in [PREFERRED_DATASET, FALLBACK_DATASET]:
        try:
            ds = load_dataset(ds_name, split="validation")
            return [dict(x) for x in ds], ds_name
        except Exception as e:
            print(f"[Info] Failed to load {ds_name} ({e}). Trying next...")

    raise RuntimeError("Could not load 2WikiMultiHopQA from Hugging Face (both preferred + fallback failed).")

records, source_used = load_2wikimultihopqa_dev()
print(f"Loaded DEV split from: {source_used}")
print(f"Type(records): {type(records).__name__} | Length: {len(records)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/165M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/29.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/28.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/167454 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/12576 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12576 [00:00<?, ? examples/s]

Loaded DEV split from: framolfese/2WikiMultihopQA
Type(records): list | Length: 12576


In [3]:
# 3: Pretty-print the first three examples (records[0:3]) to familiarize with the structure
from pprint import pprint
if not isinstance(records, list) or len(records) == 0:
    raise RuntimeError("Dataset appears to be empty or not a list. Check the previous cell output.")

for i, item in enumerate(records[:3], start=1):
    print("=" * 20, f"Sample #{i}", "=" * 20)
    pprint(item, width=120, sort_dicts=False)
    print()

==================== Sample #1 ====================
{'id': '8813f87c0bdd11eba7f7acde48001122',
 'question': 'Who is the mother of the director of film Polish-Russian War (Film)?',
 'answer': 'Małgorzata Braunek',
 'type': 'compositional',
 'evidences': [['Polish-Russian War', 'director', 'Xawery Żuławski'],
               ['Xawery Żuławski', 'mother', 'Małgorzata Braunek']],
 'supporting_facts': {'title': ['Polish-Russian War (film)', 'Xawery Żuławski'], 'sent_id': [1, 2]},
 'context': {'title': ['Maheen Khan',
                       'Viktor Yeliseyev',
                       'Alice Washburn',
                       'Minamoto no Chikako',
                       'Polish-Russian War (film)',
                       'A Snow White Christmas',
                       'Snow White and the Three Stooges',
                       'Xawery Żuławski',
                       'Snow White and the Seven Dwarfs (1955 film)',
                       'Liberty Ross'],
             'sentences': [['Maheen Khan 

In [4]:
# 4: Download Wikipedia dump (multistream) for 2020-01-01 (stream, no full unpack)

WIKI_SNAPSHOT_DATE = "2020-01-01"

# Archive.org has the full multistream bz2 for this snapshot
DUMP_URL  = "https://archive.org/download/enwiki-20200101/enwiki-20200101-pages-articles-multistream.xml.bz2"
DUMP_PATH = Path("enwiki-20200101-pages-articles-multistream.xml.bz2")

def download_large_file(url: str, dest: Path) -> None:
    """Download large file with resume (aria2c if available; else wget)."""
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[Info] Found existing: {dest} ({dest.stat().st_size/1e9:.2f} GB). Skipping download.")
        return

    if shutil.which("aria2c"):
        print("[Info] Using aria2c for download (resume + multi-conn).")
        subprocess.run([
            "aria2c", "-x", "16", "-s", "16", "-k", "1M",
            "-o", dest.name, "-d", str(dest.parent.resolve()),
            url
        ], check=True)
    else:
        print("[Info] Using wget for download (resume).")
        subprocess.run(["wget", "-c", url, "-O", str(dest)], check=True)

download_large_file(DUMP_URL, DUMP_PATH)
print(f"[Info] Dump ready: {DUMP_PATH} | Size: {DUMP_PATH.stat().st_size/1e9:.2f} GB")


[Info] Using wget for download (resume).
[Info] Dump ready: enwiki-20200101-pages-articles-multistream.xml.bz2 | Size: 17.75 GB


In [5]:
# 5: Build title-set from DEV, init SQLite, and prepare sentence splitting

def normalize_title_key(title: str) -> str:
    """Normalize titles for matching (casefold + spaces)."""
    title = html.unescape(str(title)).replace("_", " ").strip()
    return title.casefold()

def get_context_titles_and_sentences(example: Dict[str, Any]) -> Tuple[List[str], List[List[str]]]:
    """Return aligned (titles, sentences-per-title) from 2Wiki example."""
    ctx = example.get("context")
    if not isinstance(ctx, dict):
        return [], []
    titles = ctx.get("title", [])
    paras  = ctx.get("sentences", [])
    if not (isinstance(titles, list) and isinstance(paras, list)):
        return [], []
    m = min(len(titles), len(paras))

    out_titles: List[str] = []
    out_paras: List[List[str]] = []
    for i in range(m):
        t = titles[i]
        p = paras[i]
        if not isinstance(t, str):
            continue
        if not isinstance(p, list):
            p = []
        p2 = [s for s in p if isinstance(s, str)]
        out_titles.append(t)
        out_paras.append(p2)
    return out_titles, out_paras

# ---- 1) Build wanted title set from DEV
ALL_TITLES: Set[str] = set()
for ex in records:
    titles, _ = get_context_titles_and_sentences(ex)
    ALL_TITLES.update(titles)

wanted_norm_titles: Set[str] = {normalize_title_key(t) for t in ALL_TITLES}

# Canonical key per normalized title (prefer dataset spelling)
title_key_for_norm: Dict[str, str] = {}
for t in sorted(ALL_TITLES):
    title_key_for_norm.setdefault(normalize_title_key(t), t)

print(f"[Info] DEV unique titles: {len(ALL_TITLES)}")

# ---- 2) SQLite store
DB_PATH = Path("enwiki20200101_pages.sqlite")

def init_db(db_path: Path) -> sqlite3.Connection:
    conn = sqlite3.connect(str(db_path))
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("""
    CREATE TABLE IF NOT EXISTS pages (
        title_key      TEXT PRIMARY KEY,
        dump_title     TEXT,
        page_id        INTEGER,
        redirect_to    TEXT,
        docs           TEXT,
        docs2_json     TEXT,
        paras_json     TEXT
    )
    """)
    conn.commit()
    return conn

conn = init_db(DB_PATH)

def db_count(conn: sqlite3.Connection) -> int:
    return int(conn.execute("SELECT COUNT(*) FROM pages").fetchone()[0])

print(f"[Info] DB ready: {DB_PATH} | Current rows: {db_count(conn)}")

# ---- 3) Ensure NLTK resources (punkt + punkt_tab)
def ensure_nltk_sentence_resources() -> None:
    """Ensure punkt and punkt_tab exist."""
    try:
        nltk.data.find("tokenizers/punkt")
    except LookupError:
        nltk.download("punkt", quiet=True)
    try:
        nltk.data.find("tokenizers/punkt_tab/english")
    except LookupError:
        nltk.download("punkt_tab", quiet=True)

ensure_nltk_sentence_resources()

def safe_sent_tokenize(text: str) -> List[str]:
    """Sentence split with NLTK; regex fallback."""
    text = text.strip()
    if not text:
        return []
    try:
        return nltk.sent_tokenize(text)
    except LookupError:
        ensure_nltk_sentence_resources()
        try:
            return nltk.sent_tokenize(text)
        except Exception:
            pass
    except Exception:
        pass
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'])", text)
    return [p.strip() for p in parts if p.strip()]


[Info] DEV unique titles: 54957
[Info] DB ready: enwiki20200101_pages.sqlite | Current rows: 0


In [6]:
# 6: Cleaning pipeline (remove </ref>, noisy sections, normalize headings), then build docs/docs2

MONTHS = {
    "1":"January","2":"February","3":"March","4":"April","5":"May","6":"June",
    "7":"July","8":"August","9":"September","10":"October","11":"November","12":"December"
}

UNWANTED_SECTION_HEADINGS = {
    "references",
    "external links",
    "footnotes",
    "ancestors",
}

def _fmt_date(y: Optional[str], m: Optional[str], d: Optional[str]) -> Optional[str]:
    if not y:
        return None
    y = y.strip()
    m = (m or "").strip()
    d = (d or "").strip()
    if m and m in MONTHS and d.isdigit():
        return f"{MONTHS[m]} {int(d)}, {y}"
    if m and m in MONTHS:
        return f"{MONTHS[m]} {y}"
    return y

def expand_known_templates_safe(code: mwparserfromhell.wikicode.Wikicode) -> None:
    """Safely expand some templates; ignore templates already removed."""
    templates = list(code.filter_templates(recursive=True))
    for tpl in templates:
        try:
            name = str(tpl.name).strip().lower().replace("_", " ")
        except Exception:
            continue

        def gp(i: int) -> Optional[str]:
            try:
                return str(tpl.get(i).value).strip()
            except Exception:
                return None

        rep: Optional[str] = None
        if name in {"birth date", "birth date and age", "bda", "birthdate"}:
            rep = _fmt_date(gp(1), gp(2), gp(3))
        elif name in {"death date", "death date and age", "dda", "deathdate"}:
            rep = _fmt_date(gp(1), gp(2), gp(3))
        elif name in {"start date", "end date"}:
            rep = _fmt_date(gp(1), gp(2), gp(3))
        elif name in {"circa", "c.", "c"}:
            v = gp(1)
            rep = f"c. {v}" if v else None
        elif name in {"nowrap", "small"}:
            rep = gp(1)
        elif name == "lang":
            rep = gp(2) or gp(1)

        if rep is None:
            continue

        try:
            code.replace(tpl, rep)
        except ValueError:
            continue
        except Exception:
            continue

_REF_TAG_RE = re.compile(r"</\s*ref\s*>|<\s*ref\b[^>]*?/?>", flags=re.I)

def remove_residual_ref_tags(text: str) -> str:
    """Remove any leftover <ref ...>, </ref> that survived parsing."""
    if not text:
        return ""
    text = _REF_TAG_RE.sub(" ", text)
    # also handle malformed patterns like '</ref' without '>'
    text = re.sub(r"</\s*ref\b.*?$", " ", text, flags=re.I)
    return text

def _canon_heading_line(line: str) -> str:
    """Normalize a heading-like line for comparisons."""
    s = line.strip()
    s = re.sub(r"^=+\s*|\s*=+$", "", s)  # drop wiki-like == ==
    s = s.strip().strip(":").strip()
    s = re.sub(r"\s+", " ", s)
    return s.casefold()

def strip_unwanted_tail_sections(plain: str) -> str:
    """Drop everything from the first unwanted section heading to the end."""
    lines = plain.splitlines()
    out = []
    for ln in lines:
        h = _canon_heading_line(ln)
        if h in UNWANTED_SECTION_HEADINGS:
            break
        out.append(ln)
    return "\n".join(out).rstrip()

def drop_category_and_empty_lines(plain: str) -> str:
    """Remove Category: lines and trim noisy empties."""
    out_lines = []
    for ln in plain.splitlines():
        s = ln.strip()
        if not s:
            out_lines.append("")
            continue
        if s.startswith("Category:") or s.startswith("CATEGORY:"):
            continue
        out_lines.append(ln)
    text = "\n".join(out_lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def is_heading_candidate(line: str) -> bool:
    """Heuristic: short line with no sentence-ending punctuation."""
    s = line.strip()
    if not s:
        return False
    if len(s) > 60:
        return False
    if re.search(r"[.!?]$", s):
        return False
    if re.match(r"^(category:|file:|image:)", s, flags=re.I):
        return False
    if re.match(r"^https?://\S+$", s, flags=re.I):
        return False
    # keep simple-ish headings only
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9 &'’\"().,\-]{0,59}", s):
        return False
    if not re.search(r"[A-Za-z]", s):
        return False
    # do not treat unwanted headings as content headings
    if _canon_heading_line(s) in UNWANTED_SECTION_HEADINGS:
        return False
    return True

def merge_headings_into_text(plain: str) -> str:
    """
    Convert:
      Heading
      content...
    into:
      Heading: content...
    Also drops headings that have no content (next non-empty line is another heading).
    """
    lines = plain.splitlines()
    out = []
    pending = None

    i = 0
    while i < len(lines):
        s = lines[i].strip()

        if not s:
            out.append("")
            i += 1
            continue

        if is_heading_candidate(s):
            # look ahead for next non-empty
            j = i + 1
            while j < len(lines) and not lines[j].strip():
                j += 1
            if j >= len(lines):
                # heading at end -> drop
                i += 1
                continue
            if is_heading_candidate(lines[j].strip()):
                # heading with no content -> drop
                i += 1
                continue
            pending = s
            i += 1
            continue

        # normal content line
        if pending:
            s = f"{pending}: {s}"
            pending = None
        out.append(s)
        i += 1

    text = "\n".join(out)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def normalize_paragraphs(plain: str) -> List[str]:
    """Split by blank lines, and remove internal newlines inside each paragraph."""
    paras = [p.strip() for p in re.split(r"\n\s*\n+", plain) if p.strip()]
    paras2 = []
    for p in paras:
        p = p.replace("\r\n", "\n").replace("\r", "\n")
        p = p.replace("\n", " ")
        p = re.sub(r"\s+", " ", p).strip()
        if p:
            paras2.append(p)
    return paras2

def clean_plaintext_for_dataset(plain: str) -> str:
    """Full cleaning: refs, unwanted sections, categories, heading merge."""
    if not plain:
        return ""

    plain = remove_residual_ref_tags(plain)
    plain = strip_unwanted_tail_sections(plain)
    plain = drop_category_and_empty_lines(plain)
    plain = merge_headings_into_text(plain)

    # remove bracketed numeric citations like [12]
    plain = re.sub(r"\[\d+\]", "", plain)

    # final whitespace normalization
    plain = plain.replace("\u00a0", " ")
    plain = re.sub(r"[ \t]+", " ", plain)
    plain = re.sub(r"\n{3,}", "\n\n", plain)
    return plain.strip()

def wikitext_to_clean_plaintext(wikitext: str) -> str:
    """Convert wikitext to clean plain text; never crash the pipeline."""
    if not wikitext:
        return ""

    # Remove ref tags early (covers normal cases)
    wikitext = re.sub(r"<ref[^>/]*?/?>", " ", wikitext, flags=re.I)
    wikitext = re.sub(r"<ref[^>]*?>.*?</ref>", " ", wikitext, flags=re.I | re.S)

    try:
        code = mwparserfromhell.parse(wikitext)
        expand_known_templates_safe(code)
        text = code.strip_code(normalize=True, collapse=False)
    except Exception:
        text = wikitext

    text = html.unescape(text)
    text = remove_residual_ref_tags(text)

    # Keep newlines (for paragraph splitting), but clean tags/garbage
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"<[^>]+>", " ", text)  # remove any leftover HTML-like tags
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = "\n".join(line.strip() for line in text.split("\n")).strip()

    return clean_plaintext_for_dataset(text)

def page_to_docs_docs2(clean_plain: str) -> Tuple[str, List[List[str]], List[str]]:
    """
    docs  = paragraphs (joined from the SAME sentences used in docs2)
    docs2 = flat sentence list (no '\n' inside sentences)
    """
    if not clean_plain:
        return "", [], []

    paras = normalize_paragraphs(clean_plain)

    paras_sents: List[List[str]] = []
    for p in paras:
        sents = safe_sent_tokenize(p)
        # enforce: no '\n' inside sentences
        sents = [re.sub(r"\s+", " ", s.replace("\n", " ")).strip() for s in sents if s and isinstance(s, str)]
        sents = [s for s in sents if s]
        if sents:
            paras_sents.append(sents)

    flat = [s for para in paras_sents for s in para]

    # docs is built ONLY by joining docs2 sentences back by paragraph (no mismatch)
    docs = "\n\n".join(" ".join(para) for para in paras_sents) if paras_sents else "\n\n".join(paras)
    return docs, paras_sents, flat


In [7]:
# 7: Stream-extract needed pages into SQLite (using cleaned docs/docs2)

def detect_mediawiki_namespace(dump_path: Path) -> str:
    """Read root tag once to get XML namespace."""
    with bz2.open(str(dump_path), "rb") as f:
        for _, elem in etree.iterparse(f, events=("start",), recover=True, huge_tree=True):
            tag = elem.tag
            m = re.match(r"^\{(.+)\}", tag)
            return m.group(1) if m else ""
    return ""

def stream_extract_pages_to_db(
    dump_path: Path,
    conn: sqlite3.Connection,
    wanted_norm_titles: Set[str],
    title_key_for_norm: Dict[str, str],
    stop_when_all_found: bool = True
) -> None:
    """One-pass stream parse of bz2 dump; store pages whose title is in wanted set."""
    cur = conn.cursor()

    ns = detect_mediawiki_namespace(dump_path)
    page_tag = f"{{{ns}}}page" if ns else "page"
    title_tag = f"{{{ns}}}title" if ns else "title"
    ns_tag = f"{{{ns}}}ns" if ns else "ns"
    page_id_tag = f"{{{ns}}}id" if ns else "id"
    redirect_tag = f"{{{ns}}}redirect" if ns else "redirect"
    revision_tag = f"{{{ns}}}revision" if ns else "revision"
    text_tag = f"{{{ns}}}text" if ns else "text"

    print(f"[Info] Detected namespace: {ns if ns else '(none)'}")
    print(f"[Info] Starting stream parse. Needed titles (norm): {len(wanted_norm_titles)}")

    found_norm: Set[str] = set()
    total_needed = len(wanted_norm_titles)

    batch = []
    pbar = tqdm(total=total_needed, desc="Extracting needed pages", unit="page")

    with bz2.open(str(dump_path), "rb") as f:
        it = etree.iterparse(f, events=("end",), tag=page_tag, recover=True, huge_tree=True)

        for _, page in it:
            title_raw = page.findtext(title_tag)
            ns_val = page.findtext(ns_tag)

            if ns_val != "0" or not title_raw:
                page.clear()
                continue

            norm = normalize_title_key(title_raw)
            if norm not in wanted_norm_titles:
                page.clear()
                continue

            title_key = title_key_for_norm.get(norm, title_raw)

            redirect_el = page.find(redirect_tag)
            redirect_to = redirect_el.get("title") if redirect_el is not None else None

            page_id_txt = page.findtext(page_id_tag)
            page_id = int(page_id_txt) if (page_id_txt and page_id_txt.isdigit()) else None

            text = ""
            rev = page.find(revision_tag)
            if rev is not None:
                txt_el = rev.find(text_tag)
                if txt_el is not None and txt_el.text:
                    text = txt_el.text

            clean_plain = wikitext_to_clean_plaintext(text)
            docs, paras_sents, flat = page_to_docs_docs2(clean_plain)

            batch.append((
                title_key,
                title_raw,
                page_id,
                redirect_to,
                docs,
                json.dumps(flat, ensure_ascii=False),
                json.dumps(paras_sents, ensure_ascii=False),
            ))

            if norm not in found_norm:
                found_norm.add(norm)
                pbar.update(1)

            if len(batch) >= 50:
                cur.executemany("""
                    INSERT OR REPLACE INTO pages
                    (title_key, dump_title, page_id, redirect_to, docs, docs2_json, paras_json)
                    VALUES (?, ?, ?, ?, ?, ?, ?)
                """, batch)
                conn.commit()
                batch.clear()

            if stop_when_all_found and len(found_norm) == total_needed:
                break

            page.clear()
            while page.getprevious() is not None:
                del page.getparent()[0]

    if batch:
        cur.executemany("""
            INSERT OR REPLACE INTO pages
            (title_key, dump_title, page_id, redirect_to, docs, docs2_json, paras_json)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, batch)
        conn.commit()

    pbar.close()
    print(f"[Info] Done. DB rows: {db_count(conn)} | Found(norm): {len(found_norm)} / {total_needed}")

# Rebuild advice
if db_count(conn) == 0:
    stream_extract_pages_to_db(DUMP_PATH, conn, wanted_norm_titles, title_key_for_norm, stop_when_all_found=True)
else:
    print("[Warn] DB already has rows. To apply cleaning, delete the DB file and re-run Cells 5-7.")


[Info] Detected namespace: http://www.mediawiki.org/xml/export-0.10/
[Info] Starting stream parse. Needed titles (norm): 54943


Extracting needed pages:   0%|          | 0/54943 [00:00<?, ?page/s]

[Info] Done. DB rows: 54943 | Found(norm): 54943 / 54943


In [8]:
# 8: Helpers: redirect-aware page resolve + robust "contains dataset sentences" check

from collections import OrderedDict
import unicodedata

def normalize_for_match(s: str) -> str:
    """Basic normalization (casefold + whitespace)."""
    s = html.unescape(str(s))
    s = s.replace("\u00a0", " ")
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s*-\s*", "-", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.casefold()

def normalize_for_containment(s: str) -> str:
    """Strong normalization for substring containment (remove punctuation/spaces)."""
    s = normalize_for_match(s)
    s = re.sub(r"[\W_]+", "", s, flags=re.UNICODE)
    return s

# Map normalized dump_title -> title_key (lazy refresh)
TITLEKEY_BY_NORM_DUMP: Dict[str, str] = {}

def refresh_titlekey_by_norm_dump() -> None:
    """Build mapping after DB is populated."""
    TITLEKEY_BY_NORM_DUMP.clear()
    for tk, dt in conn.execute("SELECT title_key, dump_title FROM pages"):
        if dt:
            TITLEKEY_BY_NORM_DUMP[normalize_title_key(dt)] = tk

def load_page_row(conn: sqlite3.Connection, title_key: str) -> Optional[Dict[str, Any]]:
    """Load page row by title_key."""
    row = conn.execute("""
        SELECT title_key, dump_title, page_id, redirect_to, docs, docs2_json
        FROM pages WHERE title_key=?
    """, (title_key,)).fetchone()
    if not row:
        return None
    return {
        "title_key": row[0],
        "dump_title": row[1],
        "page_id": row[2],
        "redirect_to": row[3],
        "docs": row[4] or "",
        "docs2": json.loads(row[5]) if row[5] else [],
    }

def lookup_title_key(name: str) -> Optional[str]:
    """Best-effort convert any title string into our DB title_key."""
    if not name:
        return None

    # already a title_key?
    if load_page_row(conn, name) is not None:
        return name

    norm = normalize_title_key(name)

    # dataset canonical
    tk = title_key_for_norm.get(norm)
    if tk and load_page_row(conn, tk) is not None:
        return tk

    # dump title mapping (build lazily)
    if not TITLEKEY_BY_NORM_DUMP:
        refresh_titlekey_by_norm_dump()

    tk2 = TITLEKEY_BY_NORM_DUMP.get(norm)
    if tk2 and load_page_row(conn, tk2) is not None:
        return tk2

    return None

def get_resolved_page(conn: sqlite3.Connection, title_key: str, max_hops: int = 6) -> Optional[Dict[str, Any]]:
    """Follow redirects within DB and return resolved page."""
    cur = lookup_title_key(title_key)
    if cur is None:
        return None

    seen = set()
    for _ in range(max_hops):
        if cur in seen:
            break
        seen.add(cur)

        row = load_page_row(conn, cur)
        if row is None:
            return None

        tgt = row.get("redirect_to")
        if tgt:
            tgt_key = lookup_title_key(tgt)
            if tgt_key and tgt_key != cur:
                tgt_row = load_page_row(conn, tgt_key)
                if tgt_row and (tgt_row.get("docs") or tgt_row.get("docs2")):
                    cur = tgt_key
                    continue
        return row

    return load_page_row(conn, cur)

# LRU cache: resolved_title_key -> normalized full-page blob
_PAGE_BLOB_LRU = OrderedDict()
_PAGE_BLOB_MAX = 1200

def get_page_blob_norm(resolved_title_key: str) -> str:
    """Get strong-normalized full-page blob for fast containment."""
    if resolved_title_key in _PAGE_BLOB_LRU:
        _PAGE_BLOB_LRU.move_to_end(resolved_title_key)
        return _PAGE_BLOB_LRU[resolved_title_key]

    row = load_page_row(conn, resolved_title_key)
    if row is None:
        blob = ""
    else:
        blob_text = " ".join(row["docs2"]) if row.get("docs2") else (row.get("docs", "") or "")
        blob = normalize_for_containment(blob_text)

    _PAGE_BLOB_LRU[resolved_title_key] = blob
    if len(_PAGE_BLOB_LRU) > _PAGE_BLOB_MAX:
        _PAGE_BLOB_LRU.popitem(last=False)
    return blob

_ACCEPT_CACHE = OrderedDict()
_ACCEPT_CACHE_MAX = 250_000

def title_is_acceptable(conn: sqlite3.Connection, title_key: str, ds_sentences: List[str]) -> bool:
    """Accept if full-page contains ALL dataset sentences (robust containment)."""
    k = (title_key, tuple(ds_sentences))
    if k in _ACCEPT_CACHE:
        _ACCEPT_CACHE.move_to_end(k)
        return _ACCEPT_CACHE[k]

    page = get_resolved_page(conn, title_key)
    if not page:
        ok = False
    else:
        blob = get_page_blob_norm(page["title_key"])
        ok = True
        for s in ds_sentences:
            sn = normalize_for_containment(s)
            if sn and sn not in blob:
                ok = False
                break

    _ACCEPT_CACHE[k] = ok
    if len(_ACCEPT_CACHE) > _ACCEPT_CACHE_MAX:
        _ACCEPT_CACHE.popitem(last=False)
    return ok

# quick sanity (run AFTER extraction)
TARGET_TITLE = "Maheen Khan"
page = get_resolved_page(conn, TARGET_TITLE)
print("[Preview] Page exists:", bool(page))
if page:
    print("[Preview] resolved title_key:", page["title_key"])
    print("[Preview] docs2 sentences:", len(page.get("docs2", [])))
    print("[Preview] blob length:", len(get_page_blob_norm(page["title_key"])))


[Preview] Page exists: True
[Preview] resolved title_key: Maheen Khan
[Preview] docs2 sentences: 11
[Preview] blob length: 830


In [9]:
# 9: Select up to 1000 examples with per-type cap=250, then rebalance if a type has fewer.
# Notes: Scan full DEV to know max eligible per type (deterministic selection).

TARGET_TYPES = ["comparison", "inference", "compositional", "bridge_comparison"]
CAP_PER_TYPE = 250
TARGET_TOTAL = 1000

def example_is_eligible_fullpage(ex: Dict[str, Any]) -> bool:
    """Eligible if every title's resolved full-page contains its dataset sentences."""
    if ex.get("type") not in TARGET_TYPES:
        return False
    titles, paras = get_context_titles_and_sentences(ex)
    if not titles or len(titles) != len(paras):
        return False
    for t, ds_sents in zip(titles, paras):
        if not title_is_acceptable(conn, t, ds_sents):
            return False
    return True

# 1) Build eligible pools (scan all to know max)
eligible_pools: Dict[str, List[Dict[str, Any]]] = {t: [] for t in TARGET_TYPES}
total_by_type = {t: 0 for t in TARGET_TYPES}

for ex in tqdm(records, desc="Scanning DEV (build eligible pools)"):
    tp = ex.get("type")
    if tp not in TARGET_TYPES:
        continue
    total_by_type[tp] += 1
    if example_is_eligible_fullpage(ex):
        eligible_pools[tp].append(ex)

available_by_type = {t: len(eligible_pools[t]) for t in TARGET_TYPES}

print("\n[Info] Availability by type (max eligible with current DB/text):")
for t in TARGET_TYPES:
    print(f"  {t:16s} | total={total_by_type[t]:5d} | eligible_max={available_by_type[t]:5d}")

# 2) Base selection: up to CAP_PER_TYPE each
selected_by_type: Dict[str, List[Dict[str, Any]]] = {}
base_counts = {}
for t in TARGET_TYPES:
    n = min(CAP_PER_TYPE, available_by_type[t])
    base_counts[t] = n
    selected_by_type[t] = eligible_pools[t][:n]

base_total = sum(base_counts.values())
remaining = TARGET_TOTAL - base_total

print(f"\n[Info] Base selected total={base_total}. Need to fill remaining={remaining} to reach {TARGET_TOTAL}.")

# 3) Fill remainder from types that still have extra eligible beyond base
if remaining > 0:
    # donors: types with leftover
    leftovers = {t: available_by_type[t] - base_counts[t] for t in TARGET_TYPES}
    donors = [t for t in TARGET_TYPES if leftovers[t] > 0]
    donors.sort(key=lambda t: leftovers[t], reverse=True)  # take from biggest pool first

    idx = {t: base_counts[t] for t in TARGET_TYPES}  # next index in pool

    while remaining > 0:
        progressed = False
        for t in donors:
            if remaining == 0:
                break
            if idx[t] < available_by_type[t]:
                selected_by_type[t].append(eligible_pools[t][idx[t]])
                idx[t] += 1
                remaining -= 1
                progressed = True
        if not progressed:
            break

if remaining > 0:
    # Not enough eligible overall -> fail with diagnostics
    total_eligible_all = sum(available_by_type.values())
    raise RuntimeError(
        f"Not enough eligible examples overall. "
        f"Total eligible across types={total_eligible_all}, need={TARGET_TOTAL}."
    )

# 4) Finalize selected_all
selected_all: List[Dict[str, Any]] = []
for t in TARGET_TYPES:
    selected_all.extend(selected_by_type[t])

print("\nSelection summary (final):")
for t in TARGET_TYPES:
    print(
        f"  {t:16s} | selected={len(selected_by_type[t]):4d} "
        f"(cap={CAP_PER_TYPE}, eligible_max={available_by_type[t]})"
    )
print(f"\nTotal selected examples: {len(selected_all)} (should be {TARGET_TOTAL})")

Scanning DEV (build eligible pools):   0%|          | 0/12576 [00:00<?, ?it/s]


[Info] Availability by type (max eligible with current DB/text):
  comparison       | total= 3040 | eligible_max=  812
  inference        | total= 1549 | eligible_max=  356
  compositional    | total= 5236 | eligible_max= 1030
  bridge_comparison | total= 2751 | eligible_max=  840

[Info] Base selected total=1000. Need to fill remaining=0 to reach 1000.

Selection summary (final):
  comparison       | selected= 250 (cap=250, eligible_max=812)
  inference        | selected= 250 (cap=250, eligible_max=356)
  compositional    | selected= 250 (cap=250, eligible_max=1030)
  bridge_comparison | selected= 250 (cap=250, eligible_max=840)

Total selected examples: 1000 (should be 1000)


In [10]:
# 10: Attach cleaned docs/docs2 (full-page) and write 1000-example JSON in Hotpot-like schema

OUTPUT_JSON_PATH = Path("2wikimultihopqa_dev_2020wiki_1000_fullwiki_clean.json")

def _to_hotpot_supporting_facts(sf: Any) -> List[List[Any]]:
    """Convert {'title':[...], 'sent_id':[...]} -> [[title, sent_id], ...]."""
    if isinstance(sf, dict) and isinstance(sf.get("title"), list) and isinstance(sf.get("sent_id"), list):
        titles = sf["title"]
        sids = sf["sent_id"]
        m = min(len(titles), len(sids))
        out = []
        for i in range(m):
            if isinstance(titles[i], str):
                out.append([titles[i], int(sids[i]) if str(sids[i]).isdigit() else sids[i]])
        return out
    # already in list form?
    if isinstance(sf, list):
        return sf
    return []

def _to_hotpot_context(ctx: Any) -> List[List[Any]]:
    """Convert {'title':[...], 'sentences':[[...], ...]} -> [[title, [sentences]], ...]."""
    if isinstance(ctx, dict) and isinstance(ctx.get("title"), list) and isinstance(ctx.get("sentences"), list):
        titles = ctx["title"]
        sents = ctx["sentences"]
        m = min(len(titles), len(sents))
        out = []
        for i in range(m):
            t = titles[i]
            ss = sents[i]
            if isinstance(t, str) and isinstance(ss, list):
                out.append([t, [x for x in ss if isinstance(x, str)]])
        return out
    # already in list form?
    if isinstance(ctx, list):
        return ctx
    return []

def attach_docs_fields_fullpage_and_format(ex: Dict[str, Any]) -> Dict[str, Any]:
    """Attach full-page cleaned docs/docs2 aligned with context titles, and format output."""
    titles, _ = get_context_titles_and_sentences(ex)
    if not titles:
        raise ValueError("Empty titles (unexpected after filtering).")

    docs: List[str] = []
    docs2: List[List[str]] = []

    for t in titles:
        page = get_resolved_page(conn, t)
        if page is None or not page.get("docs"):
            raise KeyError(f"Missing page for title={t!r} during attachment.")
        docs.append(page["docs"])
        docs2.append(list(page["docs2"]))

    out = {
        "_id": ex.get("id"),
        "question": ex.get("question"),
        "answer": ex.get("answer"),
        "type": ex.get("type"),
        "level": ex.get("level"),
        "supporting_facts": _to_hotpot_supporting_facts(ex.get("supporting_facts")),
        "context": _to_hotpot_context(ex.get("context")),
        "titles": titles,
        "docs": docs,
        "docs2": docs2,
    }

    # Optional: keep evidences if you still need them
    if "evidences" in ex:
        out["evidences"] = ex["evidences"]

    return out

selected_with_docs = [
    attach_docs_fields_fullpage_and_format(ex)
    for ex in tqdm(selected_all, desc="Attaching cleaned full-page docs")
]

print("Final type counts:", dict(Counter(ex.get("type") for ex in selected_with_docs)))
print("Total written examples:", len(selected_with_docs))

with OUTPUT_JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(selected_with_docs, f, ensure_ascii=False, indent=2)

print(f"Wrote JSON to: {OUTPUT_JSON_PATH}")


Attaching cleaned full-page docs:   0%|          | 0/1000 [00:00<?, ?it/s]

Final type counts: {'comparison': 250, 'inference': 250, 'compositional': 250, 'bridge_comparison': 250}
Total written examples: 1000
Wrote JSON to: 2wikimultihopqa_dev_2020wiki_1000_fullwiki_clean.json


In [11]:
# 11: Consistent strict checks (new schema: _id, titles/docs/docs2)

import unicodedata

def _norm_match(s: str) -> str:
    s = html.unescape(str(s))
    s = s.replace("\u00a0", " ")
    s = unicodedata.normalize("NFKC", s)
    s = re.sub(r"\s*-\s*", "-", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.casefold()

def _norm_containment(s: str) -> str:
    s = _norm_match(s)
    s = re.sub(r"[\W_]+", "", s, flags=re.UNICODE)
    return s

def build_blob_norm(doc_text: str, doc_sents: List[str]) -> str:
    """Same blob logic: prefer docs2 joined, else docs."""
    if isinstance(doc_sents, list) and doc_sents:
        joined = " ".join(x for x in doc_sents if isinstance(x, str))
        return _norm_containment(joined)
    return _norm_containment(doc_text or "")

# For checking against dataset-provided context sentences,
# we read them from the formatted 'context' (Hotpot-like).
def _get_ds_titles_and_sents_from_context(ex: Dict[str, Any]) -> Tuple[List[str], List[List[str]]]:
    ctx = ex.get("context")
    if not isinstance(ctx, list):
        return [], []
    titles = []
    paras = []
    for item in ctx:
        if isinstance(item, list) and len(item) == 2 and isinstance(item[0], str) and isinstance(item[1], list):
            titles.append(item[0])
            paras.append([s for s in item[1] if isinstance(s, str)])
    return titles, paras

mismatches = []

for ex_idx, ex in enumerate(tqdm(selected_with_docs, desc="Consistent checking (cleaned docs2 blob)")):
    titles = ex.get("titles", [])
    docs = ex.get("docs", [])
    docs2 = ex.get("docs2", [])

    ds_titles, ds_paras = _get_ds_titles_and_sents_from_context(ex)

    if not (isinstance(titles, list) and isinstance(docs, list) and isinstance(docs2, list)):
        mismatches.append({"example_index": ex_idx, "_id": ex.get("_id"), "reason": "bad fields"})
        continue
    if len(docs) != len(titles) or len(docs2) != len(titles):
        mismatches.append({"example_index": ex_idx, "_id": ex.get("_id"), "reason": "docs/docs2 alignment issue"})
        continue

    # Only check titles that exist in dataset context
    ds_map = {t: p for t, p in zip(ds_titles, ds_paras)}

    for i, t in enumerate(titles):
        if t not in ds_map:
            continue
        doc_text = docs[i] if isinstance(docs[i], str) else ""
        doc_sents = docs2[i] if isinstance(docs2[i], list) else []
        blob = build_blob_norm(doc_text, doc_sents)

        for s in ds_map[t]:
            sn = _norm_containment(s)
            if sn and sn not in blob:
                mismatches.append({
                    "example_index": ex_idx,
                    "_id": ex.get("_id"),
                    "title": t,
                    "reason": "dataset sentence missing under containment-normalization",
                    "sentence": s
                })
                break

print(f"Total mismatches (consistent criterion): {len(mismatches)}")
if mismatches:
    print("\nShowing first 5 mismatches:")
    for m in mismatches[:5]:
        print("=" * 80)
        print(m)
else:
    print("[OK] No mismatches under the same criterion used for selection.")


Consistent checking (cleaned docs2 blob):   0%|          | 0/1000 [00:00<?, ?it/s]

Total mismatches (consistent criterion): 0
[OK] No mismatches under the same criterion used for selection.


In [12]:
# 12: Pretty-print 5 random examples from the SAVED JSON

import json
import random
from pprint import pprint

INPUT_JSON_PATH = OUTPUT_JSON_PATH  # use the new cleaned output

if not INPUT_JSON_PATH.exists():
    raise FileNotFoundError(f"Could not find: {INPUT_JSON_PATH.resolve()}")

with INPUT_JSON_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)

if not isinstance(data, list) or len(data) == 0:
    raise ValueError(f"JSON must be a non-empty list. Got type={type(data)} len={len(data) if isinstance(data, list) else 'N/A'}")

picked_indices = random.sample(range(len(data)), 5)
picked = [data[i] for i in picked_indices]

print(f"[Info] Loaded {len(data)} examples from: {INPUT_JSON_PATH}")
print("[Info] Random picked indices:", picked_indices)
print()

for k, ex in enumerate(picked, start=1):
    print("=" * 24, f"Random Sample #{k}", "=" * 24)
    print(f"_id: {ex.get('_id')}")
    print(f"type: {ex.get('type')}")
    print(f"question: {ex.get('question')}")
    print("-" * 70)
    pprint(ex, width=140, sort_dicts=False)
    print()


[Info] Loaded 1000 examples from: 2wikimultihopqa_dev_2020wiki_1000_fullwiki_clean.json
[Info] Random picked indices: [36, 50, 511, 292, 254]

======================== Random Sample #1 ========================
_id: 25ffa50e08f511ebbdaaac1f6bf848b6
type: comparison
question: Was Ralf Wohlleben or Ilya Tyapkin born first?
----------------------------------------------------------------------
{'_id': '25ffa50e08f511ebbdaaac1f6bf848b6',
 'question': 'Was Ralf Wohlleben or Ilya Tyapkin born first?',
 'answer': 'Ralf Wohlleben',
 'type': 'comparison',
 'level': None,
 'supporting_facts': [['Ralf Wohlleben', 0], ['Ilya Tyapkin', 0]],
 'context': [['Ilya Tyapkin',
              ['Ilya Tyapkin( born August 2, 1991) is a Kyrgyzstani marathon runner.',
               "He competed at the 2016 Summer Olympics in the men's marathon, in which he placed 86th."]],
             ['Henry Moore (cricketer)',
              ['Henry Walter Moore( 1849 – 20 August 1916) was an English- born first- class cricke

In [16]:
# 13: Mount Google Drive and copy JSON into /MyDrive/final_project

from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

local_json_path = str(OUTPUT_JSON_PATH)
drive_folder = "/content/drive/MyDrive/final_project"
os.makedirs(drive_folder, exist_ok=True)

drive_json_path = os.path.join(drive_folder, os.path.basename(local_json_path))
shutil.copy2(local_json_path, drive_json_path)

print(f"Copied {local_json_path} -> {drive_json_path}")


Mounted at /content/drive
Copied 2wikimultihopqa_dev_2020wiki_1000_fullwiki_clean.json -> /content/drive/MyDrive/final_project/2wikimultihopqa_dev_2020wiki_1000_fullwiki_clean.json
